In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE   # for imbalance

# ===============================
# Step 1: Load + Preprocess Data
# ===============================
def load_data():
    df = pd.read_csv("dataset.csv")

    y = df['Target'].values
    X = df.drop(columns=['UDI', 'Product ID', 'Type', 'Target', 'Failure Type']).values

    # Normalize data
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Handle imbalance
    smote = SMOTE()
    X, y = smote.fit_resample(X, y)

    return X, y


# ===============================
# AIPR MODEL
# ===============================
class AIPR:
    def __init__(self, num_antibodies=200, clone_rate=5, epochs=5):
        self.num_antibodies = num_antibodies
        self.clone_rate = clone_rate
        self.epochs = epochs

    # ---------------------------
    # TRAINING
    # ---------------------------
    def train(self, X, y):
        print("Training AIPR...")

        # Step 4: Initialize antibodies randomly
        idx = np.random.choice(len(X), self.num_antibodies, replace=False)
        self.antibodies = X[idx].copy()
        self.labels = y[idx].copy()

        # Step 5 & 6: Training loop
        for epoch in range(self.epochs):
            print(f"Epoch {epoch+1}")

            for i, antigen in enumerate(X):
                true_label = y[i]

                # Step: Affinity (distance)
                distances = np.linalg.norm(self.antibodies - antigen, axis=1)
                best_idx = np.argmin(distances)
                best_ab = self.antibodies[best_idx]

                # Step: Cloning
                clones = np.tile(best_ab, (self.clone_rate, 1))

                # Step: Adaptive Mutation (important)
                mutation_rate = 1 / (1 + distances[best_idx])
                noise = np.random.randn(*clones.shape) * mutation_rate
                mutated = clones + noise

                # Step: Select best clone
                clone_dist = np.linalg.norm(mutated - antigen, axis=1)
                best_clone = mutated[np.argmin(clone_dist)]

                # Replace if better
                if np.min(clone_dist) < distances[best_idx]:
                    self.antibodies[best_idx] = best_clone
                    self.labels[best_idx] = true_label

    # ---------------------------
    # PREDICTION
    # ---------------------------
    def predict(self, X):
        preds = []

        for antigen in X:
            distances = np.linalg.norm(self.antibodies - antigen, axis=1)
            best_idx = np.argmin(distances)
            preds.append(self.labels[best_idx])

        return np.array(preds)


# ===============================
# MAIN EXECUTION
# ===============================
if __name__ == "__main__":
    X, y = load_data()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = AIPR(num_antibodies=300, clone_rate=5, epochs=5)

    model.train(X_train, y_train)

    preds = model.predict(X_test)

    print("\nResults:")
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))

Training AIPR...
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5

Results:
Accuracy: 0.7723156532988357
              precision    recall  f1-score   support

           0       0.72      0.88      0.79      1934
           1       0.85      0.67      0.75      1931

    accuracy                           0.77      3865
   macro avg       0.79      0.77      0.77      3865
weighted avg       0.79      0.77      0.77      3865

